# BAB 2 — DATA PREPROCESSING

## 2.1 Pengenalan Data Preprocessing

Data yang digunakan untuk Machine Learning biasanya belum langsung siap digunakan.

Data dapat memiliki beberapa masalah seperti:
- Missing value
- Perbedaan skala antar fitur
- Data kategorikal
- Format data yang tidak sesuai

Data preprocessing adalah proses mempersiapkan data sebelum digunakan untuk melatih model.

Beberapa teknik preprocessing yang umum digunakan:
- Imputation
- Scaling
- Normalization
- Encoding
- Feature Engineering

## 2.2 Membuat Dataset

Pada contoh ini dibuat dataset sederhana yang memiliki fitur umur, gaji, dan kota.

Fitur umur dan gaji merupakan data numerik, sedangkan kota merupakan data kategorikal.

In [1]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    "umur": [20, 25, 30, 35, 40],
    "gaji": [3000000, 5000000, 7000000, 9000000, 12000000],
    "kota": ["Bandung", "Jakarta", "Bandung", "Surabaya", "Jakarta"]
})

data

,umur,gaji,kota
0,20,3000000,Bandung
1,25,5000000,Jakarta
2,30,7000000,Bandung
3,35,9000000,Surabaya
4,40,12000000,Jakarta


## 2.3 Missing Value

Missing value adalah kondisi ketika suatu data tidak memiliki nilai.

Missing value dapat menyebabkan masalah ketika data digunakan untuk Machine Learning.

Pada contoh berikut, nilai gaji pada salah satu baris dibuat menjadi NaN.

In [2]:
data.loc[2, "gaji"] = np.nan

data

,umur,gaji,kota
0,20,3000000.0,Bandung
1,25,5000000.0,Jakarta
2,30,NaN,Bandung
3,35,9000000.0,Surabaya
4,40,12000000.0,Jakarta


Untuk mengetahui jumlah missing value pada setiap kolom, dapat digunakan isnull().sum().

In [3]:
data.isnull().sum()

,0
umur,0
gaji,1
kota,0


## 2.4 SimpleImputer

SimpleImputer digunakan untuk mengisi nilai yang hilang.

Salah satu strategi yang dapat digunakan adalah mean.

strategy="mean"
→ missing value diganti dengan nilai rata-rata.

Strategi lain yang dapat digunakan:
- mean → rata-rata
- median → nilai tengah
- most_frequent → nilai yang paling sering muncul

In [4]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

data[["gaji"]] = imputer.fit_transform(data[["gaji"]])

data

,umur,gaji,kota
0,20,3000000.0,Bandung
1,25,5000000.0,Jakarta
2,30,7250000.0,Bandung
3,35,9000000.0,Surabaya
4,40,12000000.0,Jakarta


## 2.5 StandardScaler

StandardScaler digunakan untuk melakukan standardisasi data.

Scaling penting ketika fitur memiliki rentang nilai yang sangat berbeda.

Contohnya:

umur → 20–40
gaji → 3.000.000–12.000.000

Jika tidak dilakukan scaling, fitur dengan nilai yang lebih besar dapat lebih dominan pada algoritma tertentu.

StandardScaler mengubah skala data sehingga rata-rata mendekati 0 dan standar deviasi mendekati 1.

In [5]:
from sklearn.preprocessing import StandardScaler

X = np.array([
    [20, 3000000],
    [25, 5000000],
    [30, 7000000],
    [35, 9000000],
    [40, 12000000]
])

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[-1.41421356 -1.34438724]
 [-0.70710678 -0.70420284]
 [ 0.         -0.06401844]
 [ 0.70710678  0.57616596]
 [ 1.41421356  1.53644256]]


## 2.6 MinMaxScaler

MinMaxScaler merupakan teknik scaling yang mengubah nilai fitur ke dalam rentang tertentu.

Secara default, nilai akan berada pada rentang 0 sampai 1.

MinMaxScaler dapat digunakan ketika kita ingin mempertahankan hubungan relatif antar nilai tetapi mengubah skala datanya.

In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[0.         0.        ]
 [0.25       0.22222222]
 [0.5        0.44444444]
 [0.75       0.66666667]
 [1.         1.        ]]


## 2.7 OneHotEncoder

Data kategorikal seperti nama kota tidak dapat langsung digunakan oleh sebagian besar algoritma Machine Learning.

Contoh:

Bandung
Jakarta
Surabaya

OneHotEncoder mengubah kategori tersebut menjadi bentuk numerik.

Contohnya:

Bandung → [1, 0, 0]
Jakarta → [0, 1, 0]
Surabaya → [0, 0, 1]

In [7]:
from sklearn.preprocessing import OneHotEncoder

kota = np.array([
    ["Bandung"],
    ["Jakarta"],
    ["Bandung"],
    ["Surabaya"]
])

encoder = OneHotEncoder(sparse_output=False)

hasil = encoder.fit_transform(kota)

print(hasil)

[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


## 2.8 ColumnTransformer

Dalam dataset nyata, kita dapat memiliki beberapa jenis fitur sekaligus.

Contohnya:

umur → numerik
gaji → numerik
kota → kategorikal

ColumnTransformer memungkinkan kita memberikan preprocessing yang berbeda untuk setiap jenis kolom.

Pada contoh ini:
- Kolom numerik menggunakan StandardScaler
- Kolom kategorikal menggunakan OneHotEncoder

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

data = pd.DataFrame({
    "umur": [20, 25, 30, 35],
    "gaji": [3000000, 5000000, 7000000, 9000000],
    "kota": ["Bandung", "Jakarta", "Bandung", "Surabaya"]
})

preprocessor = ColumnTransformer([
    ("numerik", StandardScaler(), ["umur", "gaji"]),
    ("kategori", OneHotEncoder(handle_unknown="ignore"), ["kota"])
])

X_processed = preprocessor.fit_transform(data)

print(X_processed)

[[-1.34164079 -1.34164079  1.          0.          0.        ]
 [-0.4472136  -0.4472136   0.          1.          0.        ]
 [ 0.4472136   0.4472136   1.          0.          0.        ]
 [ 1.34164079  1.34164079  0.          0.          1.        ]]


## 2.9 Feature Engineering

Feature Engineering adalah proses membuat fitur baru berdasarkan fitur yang sudah tersedia.

Fitur baru dapat membantu memberikan informasi tambahan kepada model.

Contohnya, jika terdapat data harga dan jumlah barang, kita dapat membuat fitur baru berupa total harga.

Total Harga = Harga × Jumlah

In [9]:
data = pd.DataFrame({
    "harga": [100000, 200000, 300000],
    "jumlah": [2, 3, 5]
})

data["total_harga"] = data["harga"] * data["jumlah"]

data

,harga,jumlah,total_harga
0,100000,2,200000
1,200000,3,600000
2,300000,5,1500000


## 2.10 Kesimpulan Bab 2

Data preprocessing digunakan untuk mempersiapkan data sebelum masuk ke model Machine Learning.

SimpleImputer
→ Mengatasi missing value.

StandardScaler
→ Melakukan standardisasi data.

MinMaxScaler
→ Mengubah data ke rentang tertentu.

OneHotEncoder
→ Mengubah data kategorikal menjadi numerik.

ColumnTransformer
→ Melakukan preprocessing berbeda pada beberapa kolom.

Feature Engineering
→ Membuat fitur baru dari fitur yang sudah ada.

Secara sederhana:

Data Mentah
↓
Membersihkan Data
↓
Scaling
↓
Encoding
↓
Feature Engineering
↓
Data Siap Digunakan